## 1. Data Preparation and Feature Engineering



In [1]:
import os
from google.colab import drive

drive.mount('/content/drive')

data_path = '/content/drive/MyDrive/stockdata'
os.makedirs(data_path, exist_ok=True)

parquet_path = f"{data_path}/parquet_data"
os.makedirs(parquet_path, exist_ok=True)

Mounted at /content/drive


In [ ]:
# !apt-get install p7zip-full # for extracting the zip files

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


In [ ]:
!pip install tqdm patool

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.4/88.4 kB 3.0 MB/s eta 0:00:00


In [ ]:
import re
import glob
import shutil
import patoolib
import numpy as np
import pandas as pd
import tqdm.notebook as tqdm


##------- Pytorch and related stuff
import torch
import torch.nn as nn
import torch.nn.functional as f
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset
from torch.utils.data import DataLoader


In [ ]:
yiuyyuy97n # # # RUN ONCE AND COMMENT THIS BLOCK

# def extract_zip_save_parquet(file):
#   if os.path.exists(f"{data_path}/cleaned_data"):
#     shutil.rmtree(f"{data_path}/cleaned_data")

#   filename = os.path.basename(file)
#   patoolib.extract_archive(file, outdir=data_path)t5
#   print("******Done extracting******")

#   print("******Getting filenames******")
#   files = sorted(glob.glob(f"{data_path}/cleaned_data/*.csv"))

#   print(f" Glob found {len(files)} files.")

#   dataframes = {}

#   for file in tqdm.tqdm(files, desc="Loading CSVs"):
#     filename = os.path.basename(file).split('.')[0]
#     df = pd.read_csv(file)
#     df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], errors='coerce')
#     df['EXPIRY'] = pd.to_datetime(df['EXPIRY'], errors='coerce')
#     num_cols = df.columns.drop(["TIMESTAMP", "EXPIRY"])
#     df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")
#     df.to_parquet(f"{parquet_path}/{filename}.parquet")
#     print(f"*****Saved: {filename}.parquet")


# zip_path = '/content/drive/MyDrive/niftydata'

# zips = sorted(glob.glob(f"{zip_path}/*.zip"))

# for zip in tqdm.tqdm(zips, desc="Extracting and Saving Parquet"):
#   extract_zip_save_parquet(zip)

In [ ]:
# Loading parquet files
dataframes = {}

parquet_files = sorted(glob.glob(f"{parquet_path}/*.parquet"))

for file in tqdm.tqdm(parquet_files, desc="Loading Parquets"):
  filename = os.path.basename(file).split('.')[0]
  df = pd.read_parquet(file)
  dataframes[filename] = df


Loading Parquets:   0%|          | 0/211 [00:00<?, ?it/s]

Option Selling (Shorting) logic:
1. Identify Strike:
For each row (timestamp), determine the Nearest OTM (Out of The
Money) Strike.
Example: If Nifty Close is 23,475 → Nearest OTM Call is 23,500 |
Nearest OTM Put is 23,450.
2. Define Entry Price (X):
Assume we SELL the option at thePrice X = [Strike]_LOW
LOW of the current candle.
3. Define Exits:
Target: We buy back at X - 4 (Profit).
Stop Loss: We exit at X + 28 (Loss).
4. Determine Label (Simulation):
Check future candles using the HIGH price (worst-case for
shorting).

SUCCESS (1): The Option High touches X - 4 BEFORE it touches X + 28. \
FAILURE (0): The Option High touches X + 28 first, OR it never
touches the target within the trading day.

Time Weighting: The model should favor setups where the Target is hit faster.
(e.g., a trade that hits the target in 1 minute is "better" than one that takes 3
hours).


In [ ]:
df = dataframes['2024-12-31'].copy()

In [ ]:
# TODO: Ensure the logic is completely accurate!

def Label_Trades(df):
  # 1. IDENTIFY NEAREST STRIKE

  call_Strikes = (pd.Series(df.columns)
        .str.extract(r"(\d+)_CALL_")[0]
        .dropna()
        .astype(int)
        .unique()
  )

  put_Strikes = (pd.Series(df.columns)
        .str.extract(r"(\d+)_PUT_")[0]
        .dropna()
        .astype(int)
        .unique()
  )


  spot = df['NIFTY_CLOSE'].copy().to_numpy()

  call_Strikes = np.array(sorted(call_Strikes))
  put_Strikes = np.array(sorted(put_Strikes))

  call_index = np.searchsorted(call_Strikes, spot, side='right')
  put_index = np.searchsorted(put_Strikes, spot, side='left') - 1

  nearest_call = call_Strikes[np.clip(call_index, 0, len(call_Strikes) - 1)]

  nearest_put = put_Strikes[np.clip(put_index, 0, len(put_Strikes) - 1)]

  df['OTM_CALL_STRIKE'] = nearest_call.astype(int)
  df['OTM_PUT_STRIKE'] = nearest_put.astype(int)

  # 2. FIND ENTRY PRICE

  call_low_cols = df['OTM_CALL_STRIKE'].astype(str) + '_CALL_LOW'
  put_low_cols = df['OTM_PUT_STRIKE'].astype(str) + '_PUT_LOW'

  values = df.to_numpy()
  df_cols = df.columns.to_numpy()

  col_index = {c: i for i, c in enumerate(df_cols)}

  call_low_index = call_low_cols.map(col_index).to_numpy()
  put_low_index = put_low_cols.map(col_index).to_numpy()

  rows = np.arange(len(df))

  df['OTM_CALL_ENTRY'] = values[rows, call_low_index]
  df['OTM_PUT_ENTRY'] = values[rows, put_low_index]

  # 3. DEFINE EXITS

  df['OTM_CALL_TAKEPROFIT'] = df['OTM_CALL_ENTRY'] - 4
  df['OTM_CALL_STOPLOSS'] = df['OTM_CALL_ENTRY'] + 28


  df['OTM_PUT_TAKEPROFIT'] = df['OTM_PUT_ENTRY'] + 4
  df['OTM_PUT_STOPLOSS'] = df['OTM_PUT_ENTRY'] - 28 # TODO: Check if this is corrent


  # 4. DEFINE LABEL
  call_results = np.zeros(len(df), dtype=int)
  call_tt_hit = np.zeros(len(df), dtype=float)

  put_results = np.zeros(len(df), dtype=int)
  put_tt_hit = np.zeros(len(df), dtype=float)

  for i in tqdm.tqdm(range(len(df)), desc='Labeling trades:'):
    #----- Labeling calls ------
    call_tp = df['OTM_CALL_TAKEPROFIT'].iloc[i]
    call_sl = df['OTM_CALL_STOPLOSS'].iloc[i]
    call_highs = df[df['OTM_CALL_STRIKE'].iloc[i].astype(str) + '_CALL_HIGH'].iloc[i:]



    call_sl_hit = np.where(call_highs.to_numpy() >= call_sl)[0]
    call_tp_hit = np.where(call_highs.to_numpy() <= call_tp)[0]

    call_sl_hit = call_sl_hit[0] if call_sl_hit.size > 0 else np.nan
    call_tp_hit = call_tp_hit[0] if call_tp_hit.size > 0 else np.nan

    if call_tp_hit < call_sl_hit:
      call_results[i] = 1
      call_tt_hit[i] = call_tp_hit
    elif call_sl_hit == np.nan:
      call_results[i] = 0
      call_tt_hit[i] = call_sl_hit
    else:
      call_results[i] = 0
      call_tt_hit[i] = call_sl_hit

    #---- Labeling puts ------

    put_tp = df['OTM_PUT_TAKEPROFIT'].iloc[i]
    put_sl = df['OTM_PUT_STOPLOSS'].iloc[i]
    put_highs = df[df['OTM_PUT_STRIKE'].iloc[i].astype(str) + '_PUT_HIGH'].iloc[i:]

    put_sl_hits = np.where(put_highs.to_numpy() <= put_sl)[0]
    put_tp_hits = np.where(put_highs.to_numpy() >= put_tp)[0]

    put_sl_hit = put_sl_hits[0] if put_sl_hits.size > 0 else np.nan
    put_tp_hit = put_tp_hits[0] if put_tp_hits.size > 0 else np.nan


    if put_tp_hit < put_sl_hit:
      put_results[i] = 1
      put_tt_hit[i] = put_tp_hit
    elif put_sl_hit == np.nan:
      put_results[i] = 0
      put_tt_hit[i] = put_sl_hit
    else:
      put_results[i] = 0
      put_tt_hit[i] = put_sl_hit


  df['CALL_RESULT'] = call_results
  df['CALL_TT_HIT'] = call_tt_hit

  df['PUT_RESULT'] = put_results
  df['PUT_TT_HIT'] = put_tt_hit

  df = df[df.columns[~df.columns.str.contains(r'\d')]]

  return df

# ----- Testing the function ------
# df = Label_Trades(df)

Labeling trades::   0%|          | 0/22522 [00:00<?, ?it/s]

In [ ]:
for key, df in tqdm.tqdm(dataframes.items(), desc="Labeling trades"):
  df = Label_Trades(df)
  dataframes[key] = df

Labeling trades:   0%|          | 0/211 [00:00<?, ?it/s]

Labeling trades::   0%|          | 0/22522 [00:00<?, ?it/s]

Labeling trades::   0%|          | 0/22522 [00:00<?, ?it/s]

Labeling trades::   0%|          | 0/22520 [00:00<?, ?it/s]

Labeling trades::   0%|          | 0/22522 [00:00<?, ?it/s]

...

Labeling trades::   0%|          | 0/22522 [00:00<?, ?it/s]

Labeling trades::   0%|          | 0/22522 [00:00<?, ?it/s]

Labeling trades::   0%|          | 0/22522 [00:00<?, ?it/s]

Labeling trades::   0%|          | 0/22522 [00:00<?, ?it/s]

KeyError: 'NIFTY_CLOSE'

PLAN:

1. Ensure trade labeling code is accurate
2. Add technical indicators
3. Structure the data for PyTorch DataLoaders
4. Implement basic model to test data ingestion during training

## Adding Technical Indicators




Gonna add some options indicators too btw!
https://www.youtube.com/watch?v=S2OI5CwtUxg


How to choose indicators?

There are many many indicators we can use, including chart patterns.

We need indicators that are predictive or have some information content and we want to avoid putting two indicators with the same content.

We also want a wide array of indicators covering different things like market regime (risk-on-risk-off) and we want some options-based indicators to inform our choices to help manage risk exposure and avoid biting off more than we can chew.

We could start by looking at research papers to see if anyone found any indicators that are predictive for Nifty or Nifty options. Good thing is with AI we can search fast and we don’t have to spend days skimming papers like in the years before AI

We could also do some quick backtests using vectorized techniques to find good ones.

And generally, we can try to rank the indicators by some metrics and then cutoff the weak ones.

Why all this trouble? A deep learning model is relatively easy to code, but if the data is not good enough it will not learn anything. Besides, as a quant my aim is always to find the most profitable things and I spare no effort. Better this way than just picking a bunch of random indicators just because they are indicators

NOTE: I may need to do the indicator selection in a different notebook and then upload the final data here to continue. If I do that I will add link to that notebook here.

## Data Normalization and other pre-processing

## Preparing the data for PyTorch

## First Deep Learning Model